# Speech Transit Toolformer Colab demo

This notebook demonstrates the four required pipelines separately while calling project package code, CLI commands, and scripts. It does not duplicate parser, pipeline, model, dataset, or evaluation logic inside notebook cells.

## Environment setup

Use `configs/fast_model.yaml` for faster text-only testing. Use `configs/reference_model.yaml` for reference and multimodal runs.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = Path.cwd()

if IN_COLAB and not (PROJECT_ROOT / 'configs').exists():
    # In a fresh Colab runtime, clone the repository or upload it before continuing.
    raise RuntimeError('Repository files are not present. Clone or upload the project, then rerun this cell.')

if not (PROJECT_ROOT / 'configs').exists():
    raise RuntimeError('Run from the repository root.')

PYTHON = PROJECT_ROOT / '.venv' / 'bin' / 'python'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT_ROOT, check=True)

QUICK_DEMO = True
QUICK_N = 5
RUN_FULL_EVALUATION = False
print('Project root:', PROJECT_ROOT)
print('Python:', PYTHON)
print('Quick demo mode:', QUICK_DEMO)
print('Full fixed-split mode:', RUN_FULL_EVALUATION)

## Optional Google Drive mount

In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted. Copy large generated audio, predictions, or model artifacts there if needed.')
else:
    print('Drive mount skipped.')

## Load configs

In [ ]:
from src.utils.config import load_yaml_config

dataset_config = load_yaml_config('configs/dataset.yaml')
pipeline_config = load_yaml_config('configs/pipelines.yaml')
fast_pipeline_config = load_yaml_config('configs/fast_pipelines.yaml')
evaluation_config = load_yaml_config('configs/evaluation.yaml')
fast_model_config = load_yaml_config('configs/fast_model.yaml')
reference_model_config = load_yaml_config('configs/reference_model.yaml')

print('Text dataset output:', dataset_config['outputs']['text_dataset'])
print('Audio metadata output:', dataset_config['outputs']['audio_metadata'])
print('Fast text config:', fast_pipeline_config['common']['model_config_path'], fast_model_config['model']['id'])
print('Reference config:', pipeline_config['common']['model_config_path'], reference_model_config['model']['id'])

## Generate or load text dataset

In [ ]:
from src.data.loaders.jsonl import read_jsonl, write_jsonl

text_test_path = PROJECT_ROOT / dataset_config['outputs']['test']
if not text_test_path.exists():
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'generate-text-dataset'], cwd=PROJECT_ROOT, check=True)
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'validate-dataset'], cwd=PROJECT_ROOT, check=True)
else:
    print('Using existing text dataset:', text_test_path)

test_rows = read_jsonl(text_test_path)
quick_rows = test_rows[:QUICK_N]
demo_dir = PROJECT_ROOT / 'data' / 'demo'
demo_dir.mkdir(parents=True, exist_ok=True)
quick_test_path = demo_dir / 'quick_test.jsonl'
write_jsonl(quick_test_path, quick_rows)
print(f'Fixed test split rows: {len(test_rows)}')
print(f'Quick demo rows: {len(quick_rows)} -> {quick_test_path}')

## Generate or load audio dataset

In [ ]:
audio_metadata_path = PROJECT_ROOT / dataset_config['outputs']['audio_metadata']
if RUN_FULL_EVALUATION and not audio_metadata_path.exists():
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'generate-audio-dataset'], cwd=PROJECT_ROOT, check=True)
elif audio_metadata_path.exists():
    print('Using existing audio metadata:', audio_metadata_path)
else:
    print('Audio metadata is not present. Quick demo will synthesize tiny placeholder WAV files for the package pipeline runners.')

## Prepare quick demo audio and config

In [ ]:
import soundfile as sf
import numpy as np
import yaml

quick_audio_metadata_path = demo_dir / 'quick_audio_metadata.jsonl'
quick_pipeline_config_path = demo_dir / 'quick_pipelines.yaml'

if QUICK_DEMO:
    quick_audio_dir = demo_dir / 'audio' / 'test'
    quick_audio_dir.mkdir(parents=True, exist_ok=True)
    metadata_rows = []
    for row in quick_rows:
        wav_path = quick_audio_dir / f"{row['id']}.wav"
        if not wav_path.exists():
            sf.write(wav_path, np.zeros(1600, dtype=np.float32), 16000)
        metadata_rows.append({
            'audio_path': wav_path.relative_to(PROJECT_ROOT).as_posix(),
            'sample_rate': 16000,
            'language': row['language'],
            'transcript': row['user_text'],
            'duration_seconds': 0.1,
            'tts_engine': 'quick_demo_placeholder',
            'speaker_id': f"quick_{row['language']}",
        })
    with quick_audio_metadata_path.open('w', encoding='utf-8') as file:
        for metadata_row in metadata_rows:
            file.write(json.dumps(metadata_row, ensure_ascii=False) + '\n')
    quick_pipeline_config = {
        'common': {
            'dataset_path': quick_test_path.relative_to(PROJECT_ROOT).as_posix(),
            'audio_metadata_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(),
            'model_config_path': 'configs/fast_model.yaml',
            'predictions_dir': demo_dir.relative_to(PROJECT_ROOT).as_posix(),
        },
        'pipelines': {
            'A': {'name': 'text_to_tool', 'input_path': quick_test_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_a_predictions.jsonl'},
            'B': {'name': 'audio_to_transcript', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_b_predictions.jsonl'},
            'C': {'name': 'audio_to_transcript_and_tool', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'output_path': 'data/demo/pipeline_c_predictions.jsonl'},
            'D': {'name': 'audio_to_transcript_to_tool', 'input_path': quick_audio_metadata_path.relative_to(PROJECT_ROOT).as_posix(), 'transcript_output_path': 'data/demo/pipeline_d_transcripts.jsonl', 'output_path': 'data/demo/pipeline_d_predictions.jsonl'},
        },
    }
    quick_pipeline_config_path.write_text(yaml.safe_dump(quick_pipeline_config, sort_keys=False), encoding='utf-8')
    print('Quick audio metadata:', quick_audio_metadata_path)
    print('Quick pipeline config:', quick_pipeline_config_path)
else:
    print('Quick demo assets skipped.')

## Pipeline A: text -> tool call

In [ ]:
from src.models.inference.text_model import StubTextBackend, TextModelInference
from src.pipelines.pipeline_a.runner import run_pipeline_a

if QUICK_DEMO:
    responses = [json.dumps({'tool_call': row.get('expected_tool_call')}, ensure_ascii=False) if row.get('expected_tool_call') else 'No transport lookup needed.' for row in quick_rows]
    text_inference = TextModelInference(backend=StubTextBackend(responses, model_name='quick-stub-text'))
    pipeline_a_records = run_pipeline_a(input_path=quick_test_path, output_path=demo_dir / 'pipeline_a_predictions.jsonl', inference=text_inference)
else:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'run-pipeline-a', '--config', 'configs/fast_pipelines.yaml'], cwd=PROJECT_ROOT, check=True)
print('Pipeline A complete.')

## Pipeline B: audio -> transcript

In [ ]:
from src.models.inference.audio_model import AudioModelInference, StubAudioBackend, JointAudioOutput
from src.pipelines.pipeline_b.runner import run_pipeline_b

if QUICK_DEMO:
    audio_inference_b = AudioModelInference(backend=StubAudioBackend(transcript_responses=[row['user_text'] for row in quick_rows], model_name='quick-stub-audio'))
    pipeline_b_records = run_pipeline_b(dataset_path=quick_test_path, metadata_path=quick_audio_metadata_path, output_path=demo_dir / 'pipeline_b_predictions.jsonl', inference=audio_inference_b)
else:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'run-pipeline-b', '--config', 'configs/pipelines.yaml'], cwd=PROJECT_ROOT, check=True)
print('Pipeline B complete.')

## Pipeline C: audio -> transcript + tool call

In [ ]:
from src.pipelines.pipeline_c.runner import run_pipeline_c

if QUICK_DEMO:
    joint_responses = []
    for row in quick_rows:
        expected = row.get('expected_tool_call')
        joint_responses.append(JointAudioOutput(transcript=row['user_text'], raw_output=json.dumps({'tool_call': expected}, ensure_ascii=False) if expected else 'No transport lookup needed.'))
    audio_inference_c = AudioModelInference(backend=StubAudioBackend(joint_responses=joint_responses, model_name='quick-stub-audio'))
    pipeline_c_records = run_pipeline_c(dataset_path=quick_test_path, metadata_path=quick_audio_metadata_path, output_path=demo_dir / 'pipeline_c_predictions.jsonl', inference=audio_inference_c)
else:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'run-pipeline-c', '--config', 'configs/pipelines.yaml'], cwd=PROJECT_ROOT, check=True)
print('Pipeline C complete.')

## Pipeline D: audio -> transcript -> tool call

In [ ]:
from src.pipelines.pipeline_d.runner import run_pipeline_d

if QUICK_DEMO:
    audio_inference_d = AudioModelInference(backend=StubAudioBackend(transcript_responses=[row['user_text'] for row in quick_rows], model_name='quick-stub-audio'))
    text_inference_d = TextModelInference(backend=StubTextBackend(responses, model_name='quick-stub-text'))
    pipeline_d_records = run_pipeline_d(dataset_path=quick_test_path, metadata_path=quick_audio_metadata_path, output_path=demo_dir / 'pipeline_d_predictions.jsonl', audio_inference=audio_inference_d, text_inference=text_inference_d)
else:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'run-pipeline-d', '--config', 'configs/pipelines.yaml'], cwd=PROJECT_ROOT, check=True)
print('Pipeline D complete.')

## Unified evaluation

In [ ]:
from src.evaluation.benchmarks.evaluate_all import evaluate_all

if QUICK_DEMO:
    quick_eval_config_path = demo_dir / 'quick_evaluation.yaml'
    quick_eval_config = dict(evaluation_config)
    quick_eval_config['outputs'] = dict(evaluation_config['outputs'])
    quick_eval_config['outputs'].update({
        'metrics_dir': 'data/demo/metrics',
        'pipeline_a_metrics': 'data/demo/metrics/pipeline_a_metrics.json',
        'pipeline_b_metrics': 'data/demo/metrics/pipeline_b_metrics.json',
        'pipeline_c_metrics': 'data/demo/metrics/pipeline_c_metrics.json',
        'pipeline_d_metrics': 'data/demo/metrics/pipeline_d_metrics.json',
        'comparison_table': 'data/demo/metrics/comparison_table.csv',
        'failure_cases': 'data/demo/failure_cases.jsonl',
        'figures_dir': 'data/demo/figures',
    })
    quick_eval_config_path.write_text(yaml.safe_dump(quick_eval_config, sort_keys=False), encoding='utf-8')
    evaluation_outputs = evaluate_all(pipeline_config_path=quick_pipeline_config_path, evaluation_config_path=quick_eval_config_path)
elif RUN_FULL_EVALUATION:
    subprocess.run([str(PYTHON), '-m', 'src.cli', 'evaluate', '--config', 'configs/evaluation.yaml'], cwd=PROJECT_ROOT, check=True)
    evaluation_outputs = {'comparison_table': PROJECT_ROOT / evaluation_config['outputs']['comparison_table'], 'failure_cases': PROJECT_ROOT / evaluation_config['outputs']['failure_cases']}
else:
    evaluation_outputs = {}
print(evaluation_outputs)

## Show metrics tables

In [ ]:
import pandas as pd

comparison_path = Path(evaluation_outputs.get('comparison_table', PROJECT_ROOT / evaluation_config['outputs']['comparison_table']))
if comparison_path.exists():
    metrics_table = pd.read_csv(comparison_path)
    display(metrics_table)
else:
    print('No comparison table yet.')

## Show failure examples

In [ ]:
failure_path = Path(evaluation_outputs.get('failure_cases', PROJECT_ROOT / evaluation_config['outputs']['failure_cases']))
if failure_path.exists() and failure_path.stat().st_size > 0:
    failures = pd.read_json(failure_path, lines=True)
    display(failures.head(10))
else:
    print('No failure examples found, or evaluation has not been run.')

## Explain best pipeline choice

In [ ]:
if comparison_path.exists():
    metrics_table = pd.read_csv(comparison_path)
    tool_accuracy = metrics_table[metrics_table['metric'].eq('tool_exact_match_accuracy')].copy()
    wer = metrics_table[metrics_table['metric'].eq('wer')].copy()
    if not tool_accuracy.empty:
        best = tool_accuracy.sort_values('value', ascending=False).iloc[0]
        print(f"Best tool-calling pipeline: {best['pipeline']} with exact-match accuracy {best['value']:.3f}.")
    if not wer.empty:
        display(wer.sort_values('value'))
    print('For report conclusions, prefer the pipeline with the strongest tool accuracy after checking ASR WER and failure buckets. Pipeline A is the clean-text baseline; Pipelines C and D show audio-conditioned degradation and recovery.')
else:
    print('Run unified evaluation before selecting the best pipeline.')

## Full evaluation mode for the fixed test split

Set `QUICK_DEMO = False` and `RUN_FULL_EVALUATION = True` in the setup cell, then rerun the notebook. Pipeline A can use `configs/fast_pipelines.yaml` for faster text testing. Reference and multimodal runs use `configs/pipelines.yaml`, which points to `configs/reference_model.yaml`. The full mode writes report-ready outputs under `data/metrics/`, `reports/failure_cases.jsonl`, and `reports/figures/`.